# Multi-Modal RAG

Every episode so far has fed the LLM text. Vision-capable models can read images directly too — this episode generates a simple chart locally (no copyrighted artwork involved, just our own anime data plotted with matplotlib), has the model read it, then chains that visual answer into a follow-up text query against our existing anime corpus.


**Step 1 — Setup.** Quiet the logs, load API keys, and set the global LLM/embedding models used throughout this notebook.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet down noisy INFO-level logs from httpx and llama_index.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Load API keys from .env into the environment.
load_dotenv()

# Global defaults used by every index/query engine built below.
Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 2 — Generate an image to feed the model.** Plot each anime's first-aired year as a bar chart with matplotlib and save it to disk — this gives the vision model something concrete to read in the next step, with no copyrighted artwork involved.


In [2]:
import matplotlib

matplotlib.use("Agg")  # non-interactive backend so this runs headless in a notebook
import matplotlib.pyplot as plt

# Self-generated from our own anime data — no copyrighted artwork involved.
anime_years = {
    "Naruto": 2002,
    "Dragon Ball": 1986,
    "Solo Leveling": 2024,
    "Death Note": 2006,
    "Demon Slayer": 2019,
}

# A simple bar chart: one bar per anime, height = first-aired year.
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(anime_years.keys(), anime_years.values(), color="#4C72B0")
ax.set_ylabel("First Aired Year")
ax.set_title("Anime First-Aired Year Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("multimodal_chart.png", dpi=150)  # write the chart to disk as a PNG
plt.close(fig)  # free the figure now that it's saved
print("Saved multimodal_chart.png")

Saved multimodal_chart.png


**Step 3 — Have a vision-capable model read the image.** Build a `ChatMessage` that mixes a `TextBlock` (the question) with an `ImageBlock` (the chart) and send both to the LLM at once via `as_structured_llm()`, so the reply comes back as a validated `EarliestAnime` object instead of free text.


In [3]:
from pydantic import BaseModel
from llama_index.core.base.llms.types import ChatMessage, ImageBlock, TextBlock
from llama_index.llms.openai import OpenAI


# The shape we want the vision model's answer forced into.
class EarliestAnime(BaseModel):
    anime_title: str
    year: int


# as_structured_llm() makes every response get parsed into this Pydantic model.
vision_llm = OpenAI(model="gpt-4o-mini").as_structured_llm(EarliestAnime)

# A single message can carry multiple content blocks — here, one text question
# plus one image, both sent to the model in the same request.
message = ChatMessage(
    role="user",
    blocks=[
        TextBlock(text="Which anime in this chart first aired the earliest, and in what year?"),
        ImageBlock(path="multimodal_chart.png"),
    ],
)
vision_response = vision_llm.chat([message])
earliest: EarliestAnime = vision_response.raw  # the parsed EarliestAnime instance
print(f"Vision model read from the image: {earliest.anime_title} ({earliest.year})")

Vision model read from the image: Dragon Ball (1986)


**Step 4 — Chain the image-derived answer into a text query.** Use `earliest.anime_title` (extracted from the chart) as the subject of a normal text query against the existing anime corpus — this is the "multi-modal RAG" part: vision and text retrieval feeding one pipeline.


In [4]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex

# Chain the image-derived answer into a text query against the existing corpus —
# this is the "multi-modal RAG" part: vision and text retrieval feeding one pipeline.
documents = SimpleDirectoryReader("data/sample_docs").load_data()
index = VectorStoreIndex.from_documents(documents)

# Build the query text dynamically using the title the vision model just extracted.
response = index.as_query_engine().query(f"What is {earliest.anime_title}'s protagonist's signature technique?")
print(f"Q: What is {earliest.anime_title}'s protagonist's signature technique?")
print(f"A: {response}")

Q: What is Dragon Ball's protagonist's signature technique?
A: Dragon Ball's protagonist, Goku, is known for his signature technique called the Kamehameha, a concentrated beam of energy fired from cupped hands.


### Summary

- `ImageBlock` + `TextBlock` inside a `ChatMessage` is the current unified way to send images to a vision-capable model — no separate multi-modal LLM class needed, the same `OpenAI` class handles both text and images.
- Multi-modal RAG doesn't require indexing images into a vector store to be useful — here, image understanding and text retrieval were simply chained together, with the vision model's structured output driving the next query programmatically.
- `multimodal_chart.png` is generated fresh each run and is git-ignored — like `storage_demo/` and `chroma_demo/`, it's a regenerated artifact, not something to commit.
